# DuckPD Time-Series Embeddings: End-to-End Walkthrough

This notebook builds deterministic, model-free time-series representations entirely in DuckDB. It covers ordered grouped windows, representation contracts, lazy `embed_series()` execution, exact typed retrieval, and metadata-preserving Parquet persistence.

### What you will learn
- Why time-series representations require explicit row order and fixed-size windows.
- How to combine multiple semantic channels into one stable vector space.
- How centering, channel ordering, flattening, and unit normalization are declared.
- How warm-up rows, nulls, and zero-scale windows are handled.
- How a representation fingerprint prevents searches across incompatible vector spaces.
- How to persist and restore representation metadata without a model runtime.

## 1. Imports and session

The example is self-contained and offline. NumPy and pandas only create a small deterministic input fixture; all windowing, representation compilation, normalization, and retrieval run through DuckPD and DuckDB.

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pandas

import duckpd as pd

session = pd.connect(memory_limit="512MB", threads=4)
print(f"DuckPD version: {pd.__version__}")
print(f"Executions: {session.execution_count}")

## 2. Create deterministic multi-asset market data

Each ticker receives its own chronological price series. The synthetic fixture deliberately contains different trends and oscillations so nearest-window retrieval has meaningful variation while remaining reproducible.

In [ ]:
rng = np.random.default_rng(20260910)
tickers = ("NVDA", "AMD", "INTC")
bars_per_ticker = 72
records = []

for ticker_number, ticker in enumerate(tickers):
    timestamps = pandas.date_range("2026-01-05 09:30", periods=bars_per_ticker, freq="min")
    phase = np.linspace(0.0, 5.0 * np.pi, bars_per_ticker) + ticker_number * 0.7
    innovations = rng.normal(0.0, 0.0015, bars_per_ticker)
    returns = 0.0012 * np.sin(phase) + innovations
    opens = (100.0 + ticker_number * 35.0) * np.exp(np.cumsum(returns))
    closes = opens * (1.0 + returns)
    spreads = opens * (0.0015 + 0.001 * np.abs(np.cos(phase)))
    highs = np.maximum(opens, closes) + spreads
    lows = np.minimum(opens, closes) - spreads
    records.extend(
        zip(timestamps, [ticker] * bars_per_ticker, opens, highs, lows, closes, strict=True)
    )

market_data = pandas.DataFrame(
    records, columns=["timestamp", "ticker", "open", "high", "low", "close"]
)
market_data.head()

## 3. Create a lazy frame with an explicit order contract

A rolling time-series operation is undefined without order. `order_by=["ticker", "timestamp"]` declares the deterministic sequence used within each ticker. Constructing the frame copies the small fixture into the session but does not execute the analytical plan.

In [ ]:
prices = session.from_pandas(
    market_data,
    order_by=["ticker", "timestamp"],
)

print(prices)
print(f"Executions after frame construction: {session.execution_count}")

## 4. Engineer semantic channels lazily

A representation contract names channels by meaning, not by storage-column label. Here `bar_return` captures direction and `intrabar_range` captures relative volatility. Both expressions remain inside the lazy plan.

In [ ]:
features = prices.assign(
    bar_return=lambda frame: (frame["close"] - frame["open"]) / frame["open"],
    intrabar_range=lambda frame: (frame["high"] - frame["low"]) / frame["open"],
)

features[["open", "close", "bar_return", "intrabar_range"]].head(6)

## 5. Build complete grouped rolling windows

`rolling(WINDOW).to_array()` is DuckPD's fixed-window extension. It emits nullable `FLOAT[WINDOW]` arrays in oldest-first order. Grouping isolates ticker histories. The first `WINDOW - 1` rows in every ticker are null warm-up rows; DuckPD never pads an incomplete window.

In [ ]:
WINDOW = 8

windows = features.assign(
    return_window=lambda frame: frame.groupby("ticker")["bar_return"].rolling(WINDOW).to_array(),
    range_window=lambda frame: frame.groupby("ticker")["intrabar_range"].rolling(WINDOW).to_array(),
)

print(f"Window columns: return_window and range_window are FLOAT[{WINDOW}]")
print(f"Executions after window planning: {session.execution_count}")

## 6. Declare the representation space

The immutable specification is the identity of the vector space:

1. Each channel is centered independently within its window.
2. Channels are flattened in declared order, each oldest-first.
3. The concatenated vector is normalized to unit length.
4. A zero-length normalized vector becomes null instead of silently producing invalid numbers.

The output dimension is `window × channel_count`. Changing any semantic field changes the SHA-256 fingerprint.

In [ ]:
representation = pd.series_representation(
    window=WINDOW,
    channels=("bar_return", "intrabar_range"),
    sampling="observations",
    data_contract="tutorial/ohlc-shape/v1",
    normalization="center",
    unit_norm=True,
    zero_scale="null",
)

print(f"Dimension: {representation.dimension}")
print(f"Fingerprint: {representation.fingerprint}")
representation.to_dict()

## 7. Compile native embeddings lazily

`columns` maps semantic channel names to physical window columns. Mapping insertion order does not control layout; `representation.channels` does. Native representations run as DuckDB expressions—no Python row loop, model download, or inference runtime.

In [ ]:
embedded = windows.embed_series(
    columns={
        "intrabar_range": "range_window",
        "bar_return": "return_window",
    },
    into="market_shape",
    representation=representation,
)

print(f"Output dtype: FLOAT[{representation.dimension}]")
print(f"Executions after embed_series planning: {session.execution_count}")

explanation = json.loads(embedded.explain(mode="json"))
explanation["execution_boundaries"]["embedding_operations"]

## 8. Inspect warm-up and embedded rows

The first seven rows per ticker remain null because they do not contain eight observations. A non-null result is always one complete `FLOAT[16]` vector. `null_policy="propagate"` is the default: any null input window makes the whole representation null. Use `null_policy="error"` when null windows should abort execution instead.

In [ ]:
nvda_preview = embedded[embedded["ticker"] == "NVDA"][
    ["bar_return", "intrabar_range", "market_shape"]
].head(WINDOW + 2)
nvda_preview

## 9. Create a typed query from one observed window

For this tutorial, the first complete NVDA representation is the query-by-example. `EmbeddedSeriesQuery` carries both values and the representation fingerprint. DuckPD rejects an equal-width query from a different representation space before execution.

Selecting the query row is intentionally eager; corpus search remains lazy until `collect()`.

In [ ]:
nvda_candidates = embedded[(embedded["ticker"] == "NVDA") & embedded["market_shape"].notna()]
query_row = nvda_candidates[["timestamp", "market_shape"]].head(1)
query_values = tuple(float(value) for value in query_row.iloc[0]["market_shape"])
query = pd.EmbeddedSeriesQuery(query_values, representation.fingerprint)

print(f"Query endpoint: {query_row.iloc[0]['timestamp']}")
print(f"Query dimensions: {len(query.values)}")
print(f"Executions after extracting the query: {session.execution_count}")

## 10. Run exact representation-aware retrieval

The vector engine ranks complete NVDA windows by L2 distance. Because every vector has unit norm, L2 and cosine produce equivalent ordering here. `tie_breaker="timestamp"` makes equal-distance results deterministic. The query window itself appears first at distance zero.

In [ ]:
matches = nvda_candidates.vector.search(
    query,
    column="market_shape",
    metric="l2",
    k=6,
    tie_breaker="timestamp",
)[["timestamp", "close", "bar_return", "intrabar_range", "_distance"]]

print(f"Executions before collecting search: {session.execution_count}")
match_result = matches.collect()
assert match_result.iloc[0]["_distance"] == 0.0
match_result

## 11. Prove incompatible spaces fail before execution

Dimensions alone are insufficient. The following specification has the same shape but a different data contract, so its typed query must not search the existing column. This check happens during planning and performs no corpus execution.

In [ ]:
incompatible_representation = pd.series_representation(
    window=WINDOW,
    channels=("bar_return", "intrabar_range"),
    sampling="observations",
    data_contract="tutorial/different-contract/v1",
    normalization="center",
    unit_norm=True,
    zero_scale="null",
)
wrong_query = pd.EmbeddedSeriesQuery(query.values, incompatible_representation.fingerprint)
executions_before_rejection = session.execution_count

try:
    nvda_candidates.vector.search(wrong_query, column="market_shape")
except pd.errors.UnsupportedOperationError as error:
    print(f"Rejected as expected: {error}")

assert session.execution_count == executions_before_rejection

## 12. Persist vectors and representation identity

`write_parquet()` streams the lazy result directly from DuckDB. DuckPD writes a managed sidecar containing the representation contract. Reloading the Parquet file restores that identity, so the original typed query remains valid without redeclaring the specification.

In [ ]:
DEMO_DIR = Path("demo") if Path("demo").is_dir() else Path(".")
OUTPUT_DIR = DEMO_DIR / ".tmp"
OUTPUT_DIR.mkdir(exist_ok=True)
OUTPUT = OUTPUT_DIR / "time-series-embeddings.parquet"

embedded.write_parquet(OUTPUT, overwrite=True)
restored = session.read_parquet(OUTPUT)
restored_matches = (
    restored[(restored["ticker"] == "NVDA") & restored["market_shape"].notna()]
    .vector.search(
        query,
        column="market_shape",
        metric="l2",
        k=3,
        tie_breaker="timestamp",
    )[["_distance"]]
    .collect()
)

assert restored_matches.iloc[0]["_distance"] == 0.0
print(f"Persisted data: {OUTPUT}")
print(f"Persisted metadata: {OUTPUT}.duckpd-embeddings.json")
print("Typed search succeeded after reload.")

## 13. Cleanup and production checklist

For production workloads:

- Declare the true chronological order before rolling.
- Partition windows by every entity boundary that must not leak history.
- Treat `data_contract` as a versioned semantic schema.
- Persist reusable representations instead of recomputing them for repeated searches.
- Keep typed queries with their representation fingerprint.
- Expect warm-up rows to be null; do not pad incomplete history.
- Native recipes support deterministic normalization and need no model runtime. Learned series encoders are intentionally separate and are not used here.

In [ ]:
session.close()
OUTPUT.unlink(missing_ok=True)
Path(f"{OUTPUT}.duckpd-embeddings.json").unlink(missing_ok=True)
print("Session closed and tutorial artifacts removed.")